# AI Vector Databases

## Importing the libraries

In [1]:
import os
import numpy as np
import chromadb
import plotly.graph_objects as go

from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from chonkie import RecursiveChunker, OverlapRefinery
from sklearn.decomposition import PCA

## Loading NVIDIA API Key

In [2]:
load_dotenv(override = True)
api_key = os.getenv("NVIDIA_API_KEY")

if not api_key:
    print("No API key found!")
else:
    print("API key found!")

API key found!


## Defining Variables

In [3]:
DB_NAME = "Vector_DB"
DB_PATH = Path(DB_NAME)

In [4]:
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

client = OpenAI(
    base_url = NVIDIA_BASE_URL,
    api_key = api_key
)

## Creating Documents

In [5]:
files = list(Path("knowledge-base").rglob("*.md"))
print(f"Found {len(files)} files in the knowledge base")

Found 76 files in the knowledge base


In [6]:
documents = []

for file in files:
    with open(file, "r", encoding = "utf-8") as f:
        documents.append({
            "content": f.read(),
            "metadata": {
                "source": str(file),
                "doc_type": file.parent.name
            }
        })

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


## Creating Chunks

In [8]:
chunker = RecursiveChunker(
    chunk_size = 1000
)

In [9]:
refiner = OverlapRefinery(
    tokenizer = "character",
    context_size = 200,
    method = "suffix",
    merge = True
)

In [10]:
chunks = []

for document in documents:
    
    text_chunks = chunker(document["content"])
    text_chunks = refiner(text_chunks)

    for chunk in text_chunks:
        chunks.append({
            "content": chunk.text,
            "metadata": document["metadata"]
        })

print(f"Created {len(chunks)} chunks")

Created 362 chunks


## Creating Embeddings

In [12]:
def batch_embedding(texts, batch_size):

    embeddings = []

    for i in range(0, len(texts), batch_size):
        response = client.embeddings.create(
            model = "nvidia/nemotron-3-embed-1b",
            input = texts[i : i + batch_size],
            extra_body = {
                "input_type": "passage"
            }
        )
        
        embeddings.extend(item.embedding for item in response.data)

    return embeddings

In [13]:
texts = [chunk["content"] for chunk in chunks]

embeddings = batch_embedding(texts, 128)

print(f"Generated {len(embeddings)} embeddings")
print(len(embeddings[0]))

Generated 362 embeddings
2048


## Creating Vector Database

In [ ]:
client = chromadb.PersistentClient(path = str(DB_PATH))

try:
    client.delete_collection(DB_NAME)
except Exception:
    pass

collection = client.get_or_create_collection(DB_NAME)
print(f"Database Location: {DB_PATH.resolve()}")

Database Location: C:\Users\Muhammad Shahmir\Documents\AI\AI Vector Databases\Vector_DB


## Inserting Embeddings into Vector Database

In [15]:
ids = [f"Chunk_{i}" for i in range(len(chunks))]
documents = [chunk["content"] for chunk in chunks]
metadatas = [chunk["metadata"] for chunk in chunks]

collection.add(
    ids = ids,
    documents = documents,
    metadatas = metadatas,
    embeddings = embeddings
)

print(f"Vector database now contains {collection.count()} chunks.")

Vector database now contains 362 chunks.


## Defining Variables

In [16]:
result = collection.get(
    include = ['embeddings', 'documents', 'metadatas']
)

vectors = np.array(result['embeddings'])

documents = result['documents']
metadatas = result['metadatas']

doc_types = [metadata['doc_type'] for metadata in metadatas]

colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(doc_type)] for doc_type in doc_types]

## Visualising Vector Database in 2D

In [17]:
pca = PCA(n_components = 2, random_state = 42)
reduced_vectors = pca.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x = reduced_vectors[:, 0],
    y = reduced_vectors[:, 1],
    mode = 'markers',
    marker = dict(size = 5, color = colors, opacity = 0.8),
    text = [f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo = 'text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene = dict(
        xaxis_title = 'x',
        yaxis_title = 'y'
    ),
    width = 800,
    height = 600,
    margin = dict(r = 20, b = 10, l = 10, t = 40)
)

fig.show()

## Visualising Vector Database in 3D

In [18]:
pca = PCA(n_components = 3, random_state = 42)
reduced_vectors = pca.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x = reduced_vectors[:, 0],
    y = reduced_vectors[:, 1],
    z = reduced_vectors[:, 2],
    mode = 'markers',
    marker = dict(size = 5, color = colors, opacity = 0.8),
    text = [f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo = 'text'
)])

fig.update_layout(
    title = '3D Chroma Vector Store Visualization',
    scene = dict(xaxis_title = 'x', yaxis_title = 'y', zaxis_title = 'z'),
    width = 900,
    height = 700,
    margin = dict(r = 10, b = 10, l = 10, t = 40)
)

fig.show()